# 01 - Exploratory Data Analysis: CICIDS2017

**Purpose.** Inspect the raw CICIDS2017 flows, audit data quality, and gather the evidence needed to choose the cleaning and splitting strategy in notebook 02.

**Rules followed.** Raw files are never modified. Every number shown is computed by this notebook from the files in `data/raw/` (or `NIDS_RAW_DIR`). Nothing is typed in by hand.

**Run order.** Execute top to bottom. Expected runtime on the full dataset: a few minutes; peak memory roughly 2-3 GB.

In [ ]:
import sys, logging
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.config import RAW_DIR, FIGURES_DIR, EVALUATION_DIR, ensure_dirs
from src.data import loader, quality, eda
from src.data.schema import FLOW_FEATURE_COLUMNS

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_columns", 50, "display.width", 160, "display.float_format", "{:,.4f}".format)
ensure_dirs()
print("Raw data directory:", RAW_DIR)

## 1. Load data and confirm structure

In [ ]:
files = loader.list_csv_files(RAW_DIR)
for f in files:
    print(f"{f.name:<60} {f.stat().st_size/1_048_576:8.1f} MB")
df = loader.load_dataset(RAW_DIR)
layout = loader.detect_layout(df)
print()
print("Rows x columns :", df.shape)
print("Dataset layout :", layout)
print("Memory (GB)    :", round(df.memory_usage(deep=True).sum() / 1e9, 2))

In [ ]:
# Does the file match the schema this project assumes? Any mismatch must be resolved before Phase 2.
expected = set(FLOW_FEATURE_COLUMNS)
cols = set(df.columns)
print("Expected features missing from data:", sorted(expected - cols))
print("Columns in data but not in schema    :", sorted(cols - expected - {"Flow ID","Source IP","Source Port","Destination IP","Protocol","Timestamp","Label","source_file"}))
df.head(3).T.head(30)

## 2. Labels and class imbalance

In [ ]:
dist = quality.label_distribution(df)
display(dist)
imb = quality.imbalance_summary(df)
for k, v in imb.items():
    print(f"{k:<28} {v}")
display(Image(filename=str(eda.plot_label_distribution(df))))

### 2.1 Where each attack lives (capture-file structure)

CICIDS2017 was captured over five days, with specific attacks run in specific time windows. If an attack class appears in only one file, a split by file removes it from training or test entirely. This table decides which splitting strategies are even possible.

In [ ]:
by_file = quality.label_by_file(df)
display(by_file)
display(Image(filename=str(eda.plot_label_by_file(df))))

## 3. Missing, infinite and invalid values

In [ ]:
audit = quality.column_audit(df)
print(quality.invalid_value_summary(df))
print()
print("Features with NaN / inf:")
display(audit[(audit.n_missing + audit.n_pos_inf + audit.n_neg_inf) > 0][["n_missing","n_pos_inf","n_neg_inf"]])
print("Features with negative values (physically implausible for counts/durations unless a sentinel):")
display(audit[audit.n_negative > 0][["n_negative"]])
print("Constant features:", audit.index[audit.is_constant].tolist())
p = eda.plot_invalid_values(audit)
if p: display(Image(filename=str(p)))

In [ ]:
# Which labels do the invalid rows belong to? Dropping them blindly could erase a rare class.
feats = quality.feature_columns(df)
block = df[feats].to_numpy(dtype="float64", na_value=np.nan)
bad = ~np.isfinite(block).all(axis=1)
pd.DataFrame({"invalid_rows": df.loc[bad, "Label"].value_counts(),
              "total_rows": df["Label"].value_counts()}).fillna(0).astype(int).assign(
    invalid_percent=lambda t: (t.invalid_rows / t.total_rows * 100).round(3)).sort_values("invalid_rows", ascending=False)

## 4. Duplicates and label conflicts

Three separate problems:
1. **Feature duplicates** inflate scores when copies land in both train and test.
2. **Conflicting groups** (identical features, different labels) are label noise.
3. **Cross-file duplicates** break any split that assigns whole files to train or test.

In [ ]:
dup = quality.duplicate_audit(df)
for k, v in dup.items():
    print(f"{k:<32} {v}")

## 5. Correlation

In [ ]:
pairs = quality.correlated_pairs(df, threshold=0.95)
print("Feature pairs with |r| >= 0.95:", len(pairs))
display(pairs.head(25))
display(Image(filename=str(eda.plot_correlation(df))))

## 6. Identifiers, timestamps and flow structure (TrafficLabelling only)

Needed to design a leakage-aware split and to judge whether CTI enrichment is feasible.

In [ ]:
if layout == "traffic_labelling":
    print("--- Timestamps per source file ---")
    display(quality.timestamp_audit(df))
    print("--- Flow IDs ---")
    print(quality.flow_id_audit(df))
    print("--- IP audit ---")
    ips = quality.ip_audit(df)
    for col in ("Source IP", "Destination IP"):
        print(col, ips[col])
    print("Top source IPs per attack label:")
    for lab, top in ips["top_source_ips_per_attack"].items():
        print(f"  {lab:<28} {top}")
else:
    print("This copy has no IP / timestamp / Flow ID columns (MachineLearningCVE layout).")
    print("Group-aware splitting by flow endpoints and CTI enrichment are not possible with it.")

## 7. Save machine-readable summary

In [ ]:
summary = eda.run_eda(df)
print("Saved:", EVALUATION_DIR / "eda_summary.json")
print("Figures:", summary["figures"])

## 8. Findings (complete after running)

Fill these in from the outputs above. They drive Phase 2.

| Question | Answer from output |
|---|---|
| Dataset layout (TrafficLabelling or MachineLearningCVE)? | |
| Total flows / total columns? | |
| Number of classes and the smallest classes? | |
| Rows with NaN or inf, and which labels do they hit? | |
| Features with negative values? | |
| Constant features to drop? | |
| Feature-duplicate rows (count and %)? | |
| Conflicting-label groups? | |
| Cross-file duplicate vectors? | |
| Which attacks appear in only one file? | |
| Are timestamps and IPs available and parseable? | |
| Public vs private IPs (is CTI enrichment meaningful)? | |
| Number of highly correlated feature pairs? | |